<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
    </a>
</p>


# Test Environment for Generative AI classroom labs

This lab provides a test environment for the codes generated using the Generative AI classroom.

Follow the instructions below to set up this environment for further use.


# Setup


### Install required libraries

In case of a requirement of installing certain python libraries for use in your task, you may do so as shown below.


In [1]:
%pip install seaborn
import piplite

await piplite.install(['nbformat', 'plotly'])

### Dataset URL from the GenAI lab
Use the URL provided in the GenAI lab in the cell below. 


In [3]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod1.csv"


### Downloading the dataset

Execute the following code to download the dataset in to the interface.

> Please note that this step is essential in JupyterLite. If you are using a downloaded version of this notebook and running it on JupyterLabs, then you can skip this step and directly use the URL in pandas.read_csv() function to read the dataset as a dataframe


In [4]:
from pyodide.http import pyfetch

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

path = URL

await download(path, "dataset.csv")
file_name  = "dataset.csv"

---


# Test Environment


In [5]:
# Keep appending the code generated to this cell, or add more cells below this to execute in parts
import pandas as pd

file_path = "dataset.csv"
df = pd.read_csv(file_path, header=0)

<ipython-input-5-92db74e11101>:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [8]:
missing_counts = df.isnull().sum()
print(type(missing_counts))
columns_with_missing = missing_counts[missing_counts > 0].index.tolist()
print("Columns with missing values:", columns_with_missing)
print("Missing value counts per column:")
print(missing_counts)

<class 'pandas.core.series.Series'>
Columns with missing values: ['Screen_Size_cm', 'Weight_kg']
Missing value counts per column:
Unnamed: 0        0
Manufacturer      0
Category          0
Screen            0
GPU               0
OS                0
CPU_core          0
Screen_Size_cm    4
CPU_frequency     0
RAM_GB            0
Storage_GB_SSD    0
Weight_kg         5
Price             0
dtype: int64


In [9]:
# 1) Screen_Size_cm: fill missing with the most frequent value
modes = df['Screen_Size_cm'].mode()
if not modes.empty:
    df['Screen_Size_cm'] = df['Screen_Size_cm'].fillna(modes.iloc[0])

# 2) Weight_kg: fill missing with the mean
df['Weight_kg'] = df['Weight_kg'].fillna(df['Weight_kg'].mean())

In [10]:
# Convert to float dtype for the specified columns
df['Screen_Size_cm'] = df['Screen_Size_cm'].astype(float)
df['Weight_kg'] = df['Weight_kg'].astype(float)

In [11]:
# Convert and rename
df['Screen_Size_inch'] = df['Screen_Size_cm'] / 2.54
df['Weight_pounds'] = df['Weight_kg'] * 2.20462262185

# Remove old columns
df = df.drop(columns=['Screen_Size_cm', 'Weight_kg'])

In [12]:
max_val = df['CPU_frequency'].max()
if pd.notna(max_val) and max_val != 0:
    df['CPU_frequency'] = df['CPU_frequency'] / max_val
else:
    print("Normalization skipped: max value is NaN or zero.")

In [13]:
# 1) Create indicator variables for the 'Screen' column
df1 = pd.get_dummies(df['Screen'], prefix='Screen')

# 2) Append the indicator variables to the original dataframe
df = pd.concat([df, df1], axis=1)

# 3) Drop the original 'Screen' column
df = df.drop(columns=['Screen'])

In [14]:
df.head()

,Unnamed: 0,Manufacturer,Category,GPU,OS,CPU_core,CPU_frequency,RAM_GB,Storage_GB_SSD,Price,Screen_Size_inch,Weight_pounds,Screen_Full HD,Screen_IPS Panel
0,0,Acer,4,2,1,5,0.551724,8,256,978,14.0,3.527396,False,True
1,1,Dell,3,1,1,3,0.689655,4,256,634,15.6,4.850170,True,False
2,2,Dell,3,1,1,7,0.931034,8,256,946,15.6,4.850170,True,False
3,3,Dell,4,2,1,5,0.551724,8,128,1244,13.3,2.689640,False,True
4,4,HP,4,2,1,7,0.620690,8,256,837,15.6,4.210829,True,False


In [16]:
# Define the current USD to EUR exchange rate
usd_to_eur = 0.92  # replace with the actual rate

# Convert the 'Price' column from USD to EUR in place
#df['Price'] = df['Price'].astype(float) * usd_to_eur

# If you prefer to keep the original USD values, create a new column instead:
df['Price_EUR'] = df['Price'].astype(float) * usd_to_eur


In [17]:
# Ensure the column is numeric
df['CPU_frequency'] = pd.to_numeric(df['CPU_frequency'], errors='coerce')

min_val = df['CPU_frequency'].min()
max_val = df['CPU_frequency'].max()
range_val = max_val - min_val

if pd.notna(min_val) and pd.notna(max_val) and range_val != 0:
    df['CPU_frequency'] = (df['CPU_frequency'] - min_val) / range_val
else:
    print("Min-max normalization skipped: invalid min/max or zero range.")

In [19]:
df.head()

,Unnamed: 0,Manufacturer,Category,GPU,OS,CPU_core,CPU_frequency,RAM_GB,Storage_GB_SSD,Price,Screen_Size_inch,Weight_pounds,Screen_Full HD,Screen_IPS Panel,Price_EUR
0,0,Acer,4,2,1,5,0.235294,8,256,978,14.0,3.527396,False,True,899.76
1,1,Dell,3,1,1,3,0.470588,4,256,634,15.6,4.850170,True,False,583.28
2,2,Dell,3,1,1,7,0.882353,8,256,946,15.6,4.850170,True,False,870.32
3,3,Dell,4,2,1,5,0.235294,8,128,1244,13.3,2.689640,False,True,1144.48
4,4,HP,4,2,1,7,0.352941,8,256,837,15.6,4.210829,True,False,770.04


## Authors


[Abhishek Gagneja](https://www.linkedin.com/in/abhishek-gagneja-23051987/)


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-12-10|0.1|Abhishek Gagneja|Initial Draft created|


Copyright © 2023 IBM Corporation. All rights reserved.
